In [11]:
import evaluate
import numpy as np
import os
import wandb
import tempfile

from datasets import load_dataset
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    AutoTokenizer,
    AutoModelForSeq2SeqLM
    )
from dotenv import load_dotenv
import yaml
import torch
import logging

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)
model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)

num_added = tokenizer.add_special_tokens(
    {"additional_special_tokens": ["kea_Latn"]}
)

if num_added > 0:
    model.resize_token_embeddings(len(tokenizer))

kea_id = tokenizer.convert_tokens_to_ids("kea_Latn")
print(kea_id)

Loading weights: 100%|██████████| 516/516 [00:00<00:00, 14009.88it/s]


250054


In [4]:
tokenizer.save_pretrained("./mbart-kea")
model.save_pretrained("./mbart-kea")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]


In [25]:
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration

model_name = "facebook/mbart-large-50-many-to-many-mmt"

# 1. LoadTokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 2. Add the new special language token
new_token = "kea_Latn"
tokenizer.add_special_tokens({"additional_special_tokens": [new_token]})

# 3. Resize model embeddings to match the new token vocabulary size
model.resize_token_embeddings(len(tokenizer))

# 4. Map the new language token to the tokenizer's language code mappings
# This ensures tokenizer.lang_code_to_id works properly during inference/training
tokenizer.lang_code_to_id[new_token] = tokenizer.convert_tokens_to_ids(new_token)

# Choose your output directory
output_dir = "./mbart-large-50-kea_Latn"

# 1. Save the modified tokenizer (includes the new token + vocabulary mappings)
tokenizer.save_pretrained(output_dir)

# 2. Save the modified model weights & config
model.save_pretrained(output_dir)

print(f"Model and tokenizer successfully saved to {output_dir}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

Model and tokenizer successfully saved to ./mbart-large-50-kea_Latn


In [27]:
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration

model_path = "./mbart-large-50-kea_Latn"

# 1. Load saved model & tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# 2. Set source language on the tokenizer
tokenizer.src_lang = "en_XX"

# 3. Tokenize input text
text = "Hello, how are you?"
inputs = tokenizer(text, return_tensors="pt")

# 4. Get the target language token ID for your new token
tgt_token_id = tokenizer.convert_tokens_to_ids("pt_XX")

# 5. Generate translations (forcing target prefix)
generated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tgt_token_id, # Forces mBART to output in kea_Latn
    max_new_tokens=50
)

# 6. Decode output back into human-readable text
output_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

print("Input:", text)
print("Output (kea_Latn):", output_text)

Loading weights: 100%|██████████| 516/516 [00:00<00:00, 14245.21it/s]
[transformers] Both `max_new_tokens` (=50) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input: Hello, how are you?
Output (kea_Latn): Olá, como estão?


tensor([[     2, 250054,      4,    442,     25,      7,   4127,      5,      2]])